# **Correction · TP3 | Introduction au chiffrement par blocs**

À la fin de ce TP on saura :
1. chiffrer & déchiffrer des données avec AES en mode ECB (Electronic Code Book) en utilisant le module `cryptography`
2. utiliser le padding PKCS#7
3. lire et écrire des données en [Base64](https://fr.wikipedia.org/wiki/Base64)

*Corrigé rédigé en l'absence de corrigé officiel. Les réponses ont été vérifiées en Python ; les cellules qui lisent un fichier compagnon (`detect.txt`) n'ont pas pu être exécutées ici et sont à lancer dans le dossier du TP.*

## 3.1. Chiffrement AES avec `cryptography`

Chiffrement par blocs de 128 bits (16 octets), avec une clé de 128, 192 ou 256 bits (16, 24 ou 32 octets). L'algorithme est particulièrement complexe, mais il faut quand même apprendre à bien l'utiliser : en 2013, une mauvaise utilisation de chiffrement par bloc a mené à une gigantesque [fuite de mots de passe](https://github.com/thaliaarchi/adobe-cred) chez Adobe !

Le nombre de bits de sécurité de ce cryptosystème correspond à la taille de sa clé. Dans ce cas [pourquoi](https://www.reddit.com/r/AskNetsec/comments/beisil/is_aes256_indeed_more_secure_than_aes128_or_is/) aller jusqu'à 256 bits ? Peut-être la peur des ordinateurs quantiques avec lesquels le nombre de bits de sécurité seront divisés par 2.

**TODO**  
Ci-dessous on définit un cryptosystème AES avec la clé `key`. Quel est le niveau de sécurité du système ?

In [ ]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

key = b"ALL THEM WITCHES"
cipher = Cipher(algorithms.AES(key), modes.ECB())

> La clé `ALL THEM WITCHES` fait **16 octets = 128 bits** : c'est de l'AES-128.
> Le niveau de sécurité est donc de **128 bits** (2¹²⁸ clés à essayer en force brute),
> soit environ 64 bits face à un ordinateur quantique, selon l'argument du sujet.

### 3.1.a. Le chiffrement

La classe `Cipher` de `cryptography` a une méthode `.encryptor()` qui renvoie un objet `CipherContext`.

Cet objet fonctionne comme un buffer qu'on met à jour avec `.update` et qu'on *flush* avec `.finalize()`. Ces méthodes renvoient des `bytes` qu'il faut concaténer avec `+`.

**TODO**  
Chiffrer les données texte
```txt
données de test
```

In [ ]:
data = "données de test".encode()
# "données de test" fait 16 octets (le « é » en prend 2 en UTF-8) : pile un bloc AES.
encryptor = cipher.encryptor()
ciphered = encryptor.update(data) + encryptor.finalize()   # buffer : update(...) puis finalize()
print(len(ciphered), ciphered.hex())

### 3.1.b. Déchiffrement

Le déchiffrement fonctionne de la même manière, mais est créé avec `.decryptor()`.

**TODO**  
Vérifier que le message chiffré précédemment se déchiffre bien.

In [ ]:
decryptor = cipher.decryptor()
deciphered = decryptor.update(ciphered) + decryptor.finalize()
print(deciphered.decode())          # données de test
assert deciphered == data

### 3.1.c. Chiffrement de texte « quelconque »

Contrairement au chiffrement "flux" qu'on a vu aux TP précédents où les données pouvaient être de taille quelconque, AES est un chiffrement « par blocs » pour lequel le message doit être décomposé en blocs de 16 octets.

**TODO**  
Essayer de chiffrer le texte ci-dessous.
```txt
Ces données ne font pas un multiple de 16 octets.
```
Que se passe-t-il ?

In [ ]:
data2 = "Ces données ne font pas un multiple de 16 octets.".encode()
print(len(data2), "octets")         # 50 octets : 3 blocs + 2 octets qui dépassent
try:
    enc = cipher.encryptor()
    enc.update(data2) + enc.finalize()
except ValueError as err:
    print("ValueError :", err)

> On obtient une `ValueError` : *the length of the provided data is not a multiple of the block length*.
> AES ne chiffre que des blocs complets de 16 octets. Ici 50 = 3 × 16 + 2 : les 2 derniers octets ne forment pas un bloc.

## 3.2. Padding PKCS#7

Pour assurer que la taille de bloc soit respectée, on applique un **padding**, c'est-à-dire qu'on rajoute des octets à la fin des données pour atteindre un multiple de la taille de bloc.

Le méthode de padding la plus répandue est le [padding PKCS#7](https://datatracker.ietf.org/doc/html/rfc5652#section-6.3). Pour une taille de bloc `B`, elle consiste à rajouter `n` octets de valeur `n` à la fin pour que la taille des données devienne un multiple de `B`. Si la taille des données est déjà un multiple de la taille de bloc, on rajoute un bloc de `B` octets qui ont tous la valeur `B`.

Par [exemple](https://node-security.com/posts/cryptography-pkcs-7-padding/) sur des **blocs de 8 octets**, on aurait les paddings suivants :

Exemple 1  
![](padding_example1.svg)

Exemple 2  
![](padding_example2.svg)

Exemple 3  
![](padding_example_edge.svg)

**TODO (théorie)**  
Quelle est la taille de bloc maximale autorisée pour padding PKCS#7 ?

> 255 octets ! C'est beaucoup.

### 3.2.a. Fonction de padding

Le module `cryptography` implémente le padding PKCS#7 avec une syntaxe identique à celle du chiffrement, avec `.update` et `.finalize`.

**TODO**  
Utiliser le padding sur les exemples décrits précédemment.

In [ ]:
from cryptography.hazmat.primitives import padding

# Blocs de 8 octets = 64 bits : PKCS7 prend la taille de bloc EN BITS.
exemples = [b"\x01\x02\x03\x04\x05",   # 5 octets  -> 3 octets de valeur 03
            b"ABCDEFG",                  # 7 octets  -> 1 octet de valeur 01
            b"ABCDEFGH"]                 # 8 octets  -> un bloc entier de 08 (cas limite)
padded = []
for ex in exemples:
    padder = padding.PKCS7(64).padder()
    p = padder.update(ex) + padder.finalize()
    padded.append(p)
    print(ex, "->", p.hex(" "))

### 3.2.b. Fonction d'unpadding

**TODO**  
Oter le padding des exemples précédents.

In [ ]:
for p in padded:
    unpadder = padding.PKCS7(64).unpadder()
    u = unpadder.update(p) + unpadder.finalize()   # lit le dernier octet n, retire n octets
    print(p.hex(" "), "->", u)

### 3.2.c. Application du padding

**TODO**  
Utiliser ce *padding* pour chiffrer et déchiffrer le texte précédent
```txt
Ces données ne font pas un multiple de 16 octets.
```

In [ ]:
# AES travaille sur des blocs de 128 bits : PKCS7(128)
padder = padding.PKCS7(128).padder()
data2_pad = padder.update(data2) + padder.finalize()
print(len(data2_pad), "octets après padding")   # 64 = 4 blocs

enc = cipher.encryptor()
ciphered2 = enc.update(data2_pad) + enc.finalize()

dec = cipher.decryptor()
unpadder = padding.PKCS7(128).unpadder()
clair = unpadder.update(dec.update(ciphered2) + dec.finalize()) + unpadder.finalize()
print(clair.decode())

> Le padding ajoute 14 octets de valeur `0x0e` (50 + 14 = 64). Après déchiffrement, l'unpadding lit le dernier octet (14) et retire 14 octets : on retrouve le texte exact.

## 3.3. Données en Base64

L'avantage de cette écriture est qu'elle est plus compacte que l'écriture hexadécimale (4 caractères en Base64 pour 6 caractères hexadécimaux). Sa description est détaillée dans le wiki du cours.

C'est utilisé pour l'échange de clés (pour les humains) et les messages LoRaWAN de l'*Internet of Things*.

In [ ]:
from base64 import b64decode, b64encode

### 3.3.a. Encodage Base64

L'encodage se fait avec la fonction `b64encode` du module `base64`.

La sortie est une liste d'octets `bytes` qui correspond aux caractères ASCII de l'écriture en base64.

**TODO**  
On reprend l'[exemple de Wikipédia](https://en.wikipedia.org/wiki/Base64#Decoding_with_padding). Vérifier que l'encodage de `b"light work."` donne bien `b"bGlnaHQgd29yay4="`.

In [ ]:
print(b64encode(b"light work."))
print(b64encode(b"light work.") == b"bGlnaHQgd29yay4=")

**TODO (théorie)**  
En quoi est-ce différent de la méthode `bytes.hex()` ?

> Il y a deux différences : 
> 1. Un caractère en Base64 encode 6 bits alors qu'un caractère hex en encode 4.
> 2. La fonction `b64encode` renvoie un objet `bytes` alors que `bytes.hex()` renvoie un `str`.

### 3.3.b. Décodage Base64

Le décodage se fait avec `b64decode`. Un intérêt de cette fonction est qu'elle accepte un `bytes` ou un `str`.

**TODO**  
Décoder `Sm95IGFzIGFuIEFjdCBvZiBSZXNpc3RhbmNlLg==`. On devrait trouver le nom d'un album du groupe IDLES.

In [ ]:
print(b64decode("Sm95IGFzIGFuIEFjdCBvZiBSZXNpc3RhbmNlLg=="))   # b'Joy as an Act of Resistance.'

> On lit `Joy as an Act of Resistance.` : le deuxième album d'IDLES (2018).

### 3.3.c. Transfert de données chiffrées

**TODO**  
Le message ci-dessous (en base64) a été chiffré avec la clé `ALL THEM WITCHES` :
```txt
Xp24aQbySTXWvWSx5Iej10KvplSA1wJQ7WIsGKNLJUsXDbtDBSK/RxI75m+MEI+F
7JP69mDI7pP8K4vAA+Sjf7iEK4XBFSOTZEdSJF811Gq8Gso9bnU5gDP+8Dx2aORf
```
Quelle est la chanson qui correspond à ces paroles ?

In [ ]:
data = b64decode(
    """
Xp24aQbySTXWvWSx5Iej10KvplSA1wJQ7WIsGKNLJUsXDbtDBSK/RxI75m+MEI+F
7JP69mDI7pP8K4vAA+Sjf7iEK4XBFSOTZEdSJF811Gq8Gso9bnU5gDP+8Dx2aORf
"""
)
# b64decode ignore les retours à la ligne. 96 octets = 6 blocs : on peut déchiffrer.
dec = cipher.decryptor()
unpadder = padding.PKCS7(128).unpadder()
paroles = unpadder.update(dec.update(data) + dec.finalize()) + unpadder.finalize()
print(paroles.decode())

> Ce sont des paroles de **« Television »**, d'IDLES (album *Joy as an Act of Resistance.*, 2018).
> Les 96 octets font 6 blocs, et le padding retiré prouve qu'on a la bonne clé.

## Pour aller plus loin

### Détecter le mode ECB

ATTENTION le mode de chiffrement ECB qu'on utilise ici est [vraiment pourri](https://crypto.stackexchange.com/questions/20941/why-shouldnt-i-use-ecb-encryption) ! Le seul cas d'usage devrait être dans le chiffrement d'un tout petit message ([26 octets](https://www.thethingsnetwork.org/docs/lorawan/end-device-activation/#step-4-1)) dans le protocole LoRaWAN pour l'*Internet of Things*.

Ce mode de chiffrement est plein de failles. S'il est utilisé en oracle, on peut facilement déchiffrer des données et faire des injections de code (e.g. dans des cookies). Il s'agit des défis [12](https://cryptopals.com/sets/2/challenges/12), [13](https://cryptopals.com/sets/2/challenges/13) et [14](https://cryptopals.com/sets/2/challenges/14) de Cryptopals.

En plus, il est très facile de détecter si ce mode de chiffrement est utilisé, vu que tout bloc. Si le message initial est très structuré, certains blocs vont se répéter et cette répétition va se retrouver dans le chiffrement. Un bon chiffrement devrait générer des blocs qui paraissent aléatoires les uns par rapport aux autres, et donc la probabilité d'une répétition serait $\sim 2^{-128}$. Donc si un bloc se répète, on peut être sûr que le mode ECB a été utilisé.

**TODO**  
Dans le fichier `detect.txt` se trouvent les hex de messages chiffrés. Seul l'un d'eux est chiffré en mode ECB. Lequel ?

> ligne 132

(il s'agit du [défi 8](https://cryptopals.com/sets/1/challenges/8) de Cryptopals)

In [ ]:
def detect_ecb(data: bytes, block_size: int = 16) -> bool:
    """
    Detect if the given data was encrypted using ECB mode.

    Args:
        data (bytes): The input data to analyze.
        block_size (int): Block size in bytes. Default is 16 bytes (128 bits).
    Returns:
        bool: True if ECB mode is detected, False otherwise.
    """
    blocs = [data[i:i + block_size] for i in range(0, len(data), block_size)]
    # En ECB, deux blocs clairs identiques donnent deux blocs chiffrés identiques.
    # Le set élimine les doublons : s'il est plus petit que la liste, un bloc se répète.
    return len(set(blocs)) < len(blocs)

In [ ]:
with open("detect.txt", "r") as f:
    hexes = f.readlines()

for num, h in enumerate(hexes, start=1):        # numéros de ligne à partir de 1
    if detect_ecb(bytes.fromhex(h.strip())):
        print("Chiffré en ECB : ligne", num)